In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

INPUT_FILE = "/content/drive/MyDrive/Excelerate week 2/Opportunity_Analysis_Ready_FINAL.xlsx"
OUT = Path("charts_final")
OUT.mkdir(exist_ok=True)

# ---- Colour palette (muted, warm-cool balance) ---
DEEP_TEAL    = "#3A6A6E"
SAGE         = "#8FAA85"
DUSTY_PURPLE = "#7C6E86"
SOFT_BLUE    = "#6E8AA8"
CLAY         = "#B08670"
PALE_GREEN   = "#B8D0AE"
WARM_GREY    = "#57595A"
CREAM        = "#F4EFE6"

# ---- Global style ----
mpl.rcParams.update({
    "font.family":"DejaVu Sans", "font.size":11,
    "axes.titlesize":13, "axes.titleweight":"semibold", "axes.titlepad":14,
    "axes.labelsize":11, "axes.labelpad":8,
    "axes.edgecolor":WARM_GREY, "axes.linewidth":0.8,
    "axes.grid":True, "axes.axisbelow":True,
    "axes.spines.top":False, "axes.spines.right":False,
    "grid.color":"#DDD8CE", "grid.linewidth":0.6,
    "xtick.color":WARM_GREY, "ytick.color":WARM_GREY,
    "xtick.labelsize":10, "ytick.labelsize":10,
    "figure.facecolor":CREAM, "axes.facecolor":CREAM,
    "figure.dpi":150, "savefig.bbox":"tight", "savefig.facecolor":CREAM,
})

def save(fig, name, dpi=150):
    fig.savefig(OUT / f"{name}.png", dpi=dpi)
    plt.close(fig)

# ============================================================
# Load dataset
# ============================================================
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns.\n")

df["is_archived_label"]     = df["is_archived"].map({1.0:"Archived", 0.0:"Active"})
df["is_auto_approve_label"] = df["is_auto_approve"].map({1.0:"Auto-approve", 0.0:"Manual approval"})

numeric_cols = ["duration", "fee", "microscholarship"]

# =======================================================
# 1. Descriptive statistics
# =====================================================
stats = df[numeric_cols].describe().T
stats["missing"] = df[numeric_cols].isnull().sum()
stats.to_csv(OUT / "descriptive_statistics.csv")
print("Descriptive statistics:")
print(stats.round(2).to_string(), "\n")

# =========================================================
# 2. Histograms
# ========================================================
# 2.1 Duration — linear scale (well-bounded)
s = pd.to_numeric(df["duration"], errors="coerce").dropna()
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(s, bins=40, color=DUSTY_PURPLE, edgecolor=CREAM, linewidth=0.6)
ax.set_title("Distribution of Opportunity Durations")
ax.set_xlabel("Duration")
ax.set_ylabel("Number of opportunities")
ax.grid(axis="x", visible=False)
save(fig, "histogram_duration")

# 2.2 Fee — log-scale histogram + pie chart panel
s = pd.to_numeric(df["fee"], errors="coerce").dropna()
free_count = int((s == 0).sum())
paid_count = int((s != 0).sum())
s_paid = s[s > 0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.5),
                                gridspec_kw={"width_ratios":[1.7,1], "wspace":0.15})
ax1.hist(s_paid, bins=25, color=DEEP_TEAL, edgecolor=CREAM, linewidth=0.6)
ax1.set_xscale("log")
ax1.set_title("Fee — non-dominant values (log)")
ax1.set_xlabel("Fee (log scale)")
ax1.set_ylabel("Number of opportunities")
ax1.grid(axis="x", visible=False)
wedges, texts, autotexts = ax2.pie(
    [free_count, paid_count],
    labels=[f"Free\n({free_count:,})", f"Paid\n({paid_count:,})"],
    colors=[DEEP_TEAL, CLAY], startangle=90, autopct='%1.1f%%', pctdistance=0.72,
    wedgeprops=dict(edgecolor=CREAM, linewidth=1.8),
    textprops=dict(color=WARM_GREY, fontsize=9),
)
for at in autotexts:
    at.set_color(CREAM); at.set_fontweight("semibold"); at.set_fontsize(10)
ax2.set_title("Fee split")
save(fig, "histogram_fee", dpi=180)

# 2.3 Microscholarship — log-scale histogram + pie chart panel
s = pd.to_numeric(df["microscholarship"], errors="coerce").dropna()
std_count   = int((s == 120).sum())
other_count = int((s != 120).sum())
s_other = s[s != 120]
s_other = s_other[s_other > 0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.5),
                                gridspec_kw={"width_ratios":[1.7,1], "wspace":0.15})
ax1.hist(s_other, bins=25, color=SAGE, edgecolor=CREAM, linewidth=0.6)
ax1.set_xscale("log")
ax1.set_title("Microscholarship — non-dominant values (log)")
ax1.set_xlabel("Microscholarship (log scale)")
ax1.set_ylabel("Number of opportunities")
ax1.grid(axis="x", visible=False)
wedges, texts, autotexts = ax2.pie(
    [std_count, other_count],
    labels=[f"Standard (120)\n({std_count:,})", f"Other\n({other_count:,})"],
    colors=[SAGE, DUSTY_PURPLE], startangle=90, autopct='%1.1f%%', pctdistance=0.72,
    wedgeprops=dict(edgecolor=CREAM, linewidth=1.8),
    textprops=dict(color=WARM_GREY, fontsize=9),
)
for at in autotexts:
    at.set_color(CREAM); at.set_fontweight("semibold"); at.set_fontsize(10)
ax2.set_title("Microscholarship split")
save(fig, "histogram_microscholarship", dpi=180)

# ==========================================================
# 3. Outlier Analysis — portrait 2+1 boxplot layout
# ============================================================
fig = plt.figure(figsize=(10, 11))
gs  = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.6)
axes = [fig.add_subplot(gs[0, 0:2]),
        fig.add_subplot(gs[0, 2:4]),
        fig.add_subplot(gs[1, 1:3])]
configs = [("duration", DUSTY_PURPLE, "Duration", False),
           ("fee", DEEP_TEAL, "Fee", True),
           ("microscholarship", SAGE, "Microscholarship", True)]
for ax, (col, colour, title, use_log) in zip(axes, configs):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if use_log:
        s = s[s > 0]
    ax.boxplot(s, vert=True, widths=0.5, patch_artist=True,
        boxprops=dict(facecolor=colour, edgecolor=WARM_GREY, linewidth=1.0),
        medianprops=dict(color=CREAM, linewidth=2.2),
        whiskerprops=dict(color=WARM_GREY, linewidth=1.0),
        capprops=dict(color=WARM_GREY, linewidth=1.0),
        flierprops=dict(marker="o", markerfacecolor=colour, markeredgecolor=WARM_GREY,
                        markersize=5, alpha=0.45))
    if use_log:
        ax.set_yscale("log"); ax.set_ylabel("Value (log scale)")
    else:
        ax.set_ylabel("Value")
    ax.set_title(title)
    ax.set_xticks([])
    ax.grid(axis="x", visible=False)
fig.suptitle("Outlier Analysis — Numeric Variables", fontsize=16, fontweight="semibold", y=0.965)
save(fig, "boxplots_numeric", dpi=180)

# ===========================================================
# 4. Correlation heatmap
# ============================================================
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
cmap = sns.blend_palette([CREAM, SAGE, DEEP_TEAL], as_cmap=True)
sns.heatmap(corr, annot=True, cmap=cmap, center=0, vmin=-1, vmax=1,
            annot_kws={"fontsize":11, "color":WARM_GREY},
            linewidths=1.2, linecolor=CREAM,
            cbar_kws={"shrink":0.75}, ax=ax, square=True)
ax.set_title("Correlation Between Numeric Fields")
ax.tick_params(colors=WARM_GREY)
save(fig, "correlation_heatmap")

# ==========================================================
# 5. Frequency charts (categorical fields)
# ==========================================================
def frequency(col, title, colour, add_labels=False):
    s = df[col].dropna().astype(str)
    counts = s.value_counts()
    fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(counts))))
    bars = ax.barh(counts.index[::-1], counts.values[::-1],
                   color=colour, edgecolor=CREAM, linewidth=0.6)
    ax.set_title(title)
    ax.set_xlabel("Number of opportunities")
    ax.grid(axis="y", visible=False)
    if add_labels:
        xmax = counts.max()
        for bar, value in zip(bars, counts.values[::-1]):
            ax.text(value + xmax*0.008, bar.get_y() + bar.get_height()/2,
                    f"{value:,}", va="center", ha="left", fontsize=9, color=WARM_GREY)
        ax.set_xlim(0, xmax * 1.09)
    save(fig, f"frequency_{col}")

frequency("category",              "Opportunity Categories",                    DEEP_TEAL,    add_labels=True)
frequency("currency_type",         "Currencies in Use",                         SAGE)
frequency("duration_type",         "Duration Units",                            DUSTY_PURPLE)
frequency("location",              "Opportunity Locations",                     SOFT_BLUE,    add_labels=True)
frequency("is_archived_label",     "Archive Status (Available Records Only)",   CLAY)
frequency("is_auto_approve_label", "Approval Type",                             WARM_GREY)


Loaded 5,730 rows × 20 columns.

Descriptive statistics:
                   count    mean      std  min    25%    50%    75%       max  missing
duration          5721.0   24.54    16.59  0.0   12.0   30.0   30.0     359.0        9
fee               5726.0   57.45  2792.12  0.0    0.0    0.0    0.0  200000.0        4
microscholarship  5725.0  175.66   832.73  0.0  100.0  120.0  120.0   50000.0        5 

